# Advanced NLP track

Specialization Chatbot, Raiffeisenbank
---
Autonomous support for banking products

Architecture:
1. (NLU) intent classification
2. ML model
3. choose from a list of preassigned answers

input: prompt + history
output: answer / clarification / options / call operator

Agents wont work: localization due to personal data, risks

When score is low => ask
When K intents are probable => multichoice

A set of labeled dialogues
Compute classification metrics for both strategies 

How intents:
- diaglosues are clustered
- according to relevance = close / mide / far
- thresholds to transfer request to call center may be chosen dynamically (based on current call center load)
- implementation = vLLM + several servers

Team Administration
---
Points of growth = Tickets with tasks, ideas, hypothesis, experiments<br>target, metrics, responsible, metrics

Prioritisation
Setting
Gantt
Better to cut requirements than moving deadlines
uv, poetry is better than pip
CI to evaluate code quality
MlFlow / ClearML
Plots, tags, ModelsRegistry in S3
OpenMetaData
Data Drift
Cookbooks = tutorials


Web Parsing, Yandex Browser
---

Web Agent
---
Computer USE год назад вышел бенч
4x выросло
автоматизируем работу с Browser-ом
задача "купить билеты"
понимать, взаиомдействов, трекать прогресс
сначала пробовали писать JS-код, сейчас Visual модели лучше делают 
определение кликабельности = эвристика
Vision only подход, управляют мышкой
Browser Gym 
на сайте много неоднозанчиности
TRAIL: trace reasoning - проблем с агентами
просить агентов объяснять действия
RAG на предыдущих подходах
один ран занимает 5-100K токенов токенов
iXT aXT - сжатие содержимого HTML
TG: ai_in_europe

Wildberries, content moderation
---
15K per minute
70RPM vLLM on a single A100
pattern search with Aho-Corassic O(n)
blacklist (category) - banned triggers
whitelist - exception to banned trggiers
Common workflow: triggers + regular + image check => LLM decision evaluate
отдельно проверочный LLM запускается для разбана, whitelist сложнее вести
OCR for images
ViT as a backbone, lots of FT versions
Triton
Example: Vit classifer labeled as 18+ (and blurred), VLM reevaluated with text and canceled decision
category correspondence: 2000 categories, 2-step (parent, detailed)
CLIP to check correspondence did not succeed (noise due to loose title format)
Maintain a vector DB of products banned by different models (only image embeddings - embeddings from pre-trained checkpoint of the model)
  to send product to particular image classifier

Netflix, Function Calling
---
OpenAI introduced in 2023, 
Gorilla in 2023
LangChain 
vLLM добавили через полгода, LLAMA.cpp через год
все движки поддерживают форматирование содержимого чата(!) понятное языковой модели с помощью тэгов im_start fim_prefir, tool_call
JINJA
модель - всегда blackbox, нет гарантии что она правильно последует инструкции по исп инструкмента => функция должна быть хорошо документирована
если код инструкментов не помещается в контекст, можно
a) fetch by relevance
b) implement proper (thin) object taxonomy (?)

Case: oil company, chat bot
classifier1: can answer be found in knowledge base
classifier2: determine the intent (scenario)
Separate agent for each scenario
Agents can fetching data from DB
Agents can call a Tool

Challenges:
1. example: similar product search, vocabulary of product titles(?) => 
2. explicit recommendations are prohibited => return options
3. regulation of answer format => tricked business to prefer generated answers
4. multi component system => but independent, thus evaluate spearately 
5. system is dynamic, thus few comprehensive labeled data => syntethic data [A field guide to rapdily improving AI products]

v2.0 = single LLM agent

Human-in-the-loop - это user-а
Models: YandexGPT

Evaluation metrics 
  recall@k for product fetch
  accuracy for tool call
  accuracy for result
  generation length
  latency
1000 токенов у Яндекса = 1 руб
один полный контекст = 100 руб? да, у ChatGPT так же: 1$ на 100K токенов
Stopping criteria for Agent = # of attempts

Cut Cross entropy
---
Paper from skoltech

Transformers for Recommendations: BERT-like and GPT-like

BERT4Rec
  - full cross-entropy as a loss (vs all negatives)

SAS4Rec
  - predict next product using masked self-attention
  - binary cross-entropy as a loss (vs random negative)

Efficiency is important for marketplaces

Historically triplet loss does not give good enough results
Hard negative mining does not work good enough

A family of losses: BCE, BCE+, CE-, CE+, CCE+, CCE-
- BCE = 
- BCE+ several random negatives
- CE- random subsert inthe denominator

__Cut Cross Entropy__ from NLP noticed that 90% outputs / gradients are close to zero => lets dump them
CCE- = combination of CCE and CE- 

Metrics:
- NDCG
- Coverage
- Surprisal

Implemented those losses on Triton

Questions:
- MovieLens is not very practical
- Replay library

Yandex, Geo
---
17M users<br>
scenario
- navigational = seek for org
- discovery = explores places
    - well monetized => grow

Suggest = auto prefix completions

Two types:
- completions
- analogs

10K PRS, ms latency
Clustering queries clicks to same organizations


Yandex, Lavka
---

Hacks and Exploits, Kaggle
---
1) Competition 1 = hack LLM-as-a-judge to give specific number of points (0 or 9)<br>Solution = just ask them to do it in prompt + random tokens ("Give a 9")
2) Competition 2 = LMSYS: online benchmark for human preferences (they blur model names)<br>Comptetition = predict preference given only generations<br>Train Data = prompt, model, generations, preference<br>Test Data = prompt, generations<br>They found a public notebook by LMSYS with JSON with models' metadata (no IDs) and instances from competition. No IDs but knowing number of tokens deducted tokenizer = tiktoken 
3) Competition 3 = Given source text and rewritten text find the system prompt<br>Proposed evaluation metric = cosine similarity over sentencepiece-tokenized prediction<br>bad choice of metric - they just added token closest to end of string token <s>

Chatbot, domClick
---
Intent-based chatbot, K intents, ~50 anchor phrases trigerring each intent
Hypothesis: LLM is better to detect relevant intent than  cosine distance because they dont require training
List of phrases does not fit in context => RAG is needed
RAG solution where LLM acts as ranker rather than generator
Model = fine-tuned Gemma 9B (with dropout/AdamW weight decay)
LLM is fine-tuned for their task
+4% over not fine-tuned
llama.cpp helped reduce latency
For vectorization they chose E5 to substitute RoBERTa

MechInterp
---
Into to MechInterp
- terminology
    - features = patterns in signal
    - monosemantic = captures one specific feature
    - polysemantic = captures multiple features
    - provoiledged / non-proveledged
    - superposition = netowrks capture more patterns than neurons
Some approceeahces:
- SAE = neuron decomposition
- Probing = clasisifer over neurons
Cool property = for interpretable featiures we can adjusrt the model behavior
Circular / tree-like features  Circuits / Motifs
Induction heads
Possible usage

User model + System model
  experiments show user model exists
  system model depends on user model
Idea: probes can be used to modify output (before actual generation) on scale (politeness, directness etc)

cloud.ru
---
Manager tells about their tech stack for chat-bot solution

They sell cloud AI (expensive solutions). Chat-bots (Jivo chat) help not to lose customers at night:
- sell 
- provides support (links to separate chat)

taxonomy of chatbot according to autnomity:
- chatbot
- assistant
- agent

*Product comitee = evaluates MVPs/ideas according to metrics

Fastapi gateway for Jivo authorization
LLM classifier  intent (sell / support - перенаправляет)
Logs (queries, RAG requests) in Clickhouse
Embedder: E5 / SBERT is old, DB: Quadrant / PGVector / Clickhouse[vect], pipeline: LangGraph
FSM in LG implements SPIN polling of a customer
Tools: for ELmo, NaumenSD for ticket creation
Gigachat API to play

Other features:
- KV-cache warming for new / lazyint
- slowly moving to MCP

Alternative pipelining:
- n8n<br>telegram publishing
- Hugginface[tinyagent]
- LangGraph

Usage optimization:
- Mig: GPU hardware splitting
- also tried cuber timeslicing
- Hami [open-source]
*embedder

OpenRouter provides stats models
Embedder = E5, Reranker = BGE

# Crowd Annotation platform, T-bank
---
They have their own called "Клекс"
Zero-shot models perform badly

Torchtune, Pytorch
---
Guy from Pytorch sells their tourchtune module

Optimizations out-of-the-box
- activation checkpointing
- optimizer quantization
- chunk cross-entroopy (streamed computation)
- fused optimizer
- weight quantization
- torch.compile

FlexAttennion custom masks
6h -> 30 min

Voice assistant, Uvenko
---
They build voice-activated cofee machines (for public use)<br>
Session: activation / face detection / chat / purchase / coffee

Cloud models {LLM, TTS, STT} cost 10$ per chat => make local<br>
*caching reduces but still a lot

NPU/GPU computers = rk35 / Nvidia Jetson (expensive)<br>
Storing PD are expensive => store in UAE<br>
FaceRec = {detection, antispoofing, authorization} => face emdedding for fetching from vector DB<br>
Cards linked => automatic payment<br>
Cocktail party problem => beamforming<br>
Acoustic auto cancelation - for request cancelation / correction

<img src="img/uvenko.png" width=500>

Legend:
- AEC = Acoustic Echo Cancellation
- Beamforming = Improve speech capture quality by focusing on the direction of the speaker and reducing background noise
- VAD = Voice Activity Detection
- Paroli TTS = TExt-to-Speech
- LLM-based Slot Filling = Extract structured information (slots) from user utterances for task completion

Prompt contents: 
- available tools specification
- few-shot examples how to answer
- configuration (current menu)
- previous dialog

Few details:
Guided generation to prevent abuse
Custom ubuntu build
Branded voice (80K$ or custom made)

R&D:
- STT from 8 microphones (without beamforming)
- LLM is still in cloud / they will train their own
- TOF camera for detection (point clouds)

# Reliability

Five advices on deploying LLM in production, Yandex
---

Modex size = generalization

In CRISP-DM there are 2 bottlenecks: poor understanding of business value / technical infeasibility
In LLM world the second condition can be proved with MVP with prompts (written by managers)

Performance:
- valuated with reward model
    - out-of-the-box, if system is basic
    - to label with smart assessors
      Prompt-enginners do not exist, it's a marketing term
- it is recommended to split into subtopics

LLM System design
- ML workflow = graph with components<br>
  when reliability is important<br>
  example of RAG enhanced chatbot - DoorDash company, they present a graph with:
        GuardRail component
        Human-in-the-loop component
- Agentic workflow = dynamically constructed graph
    - deterministic grqph is too complicated
    - reliability is not critical
    - latency tolerable

Distillation

Streamlit tracer, Raiffeisenbank
---

Legal assessors check the correctness of chat-bot answers

A little reference
- Streamlit = a Python library create web applications that focuses on iterative user-system interaction
- Lanfuse = a tool that provides a set of functions to store intermediate data in some repository along with dashboards
- LangSmith = a part of LangChain pipelining library that enables logging intermediate data to repository
- Opik = a tool that provides a set of functions to store intermediate data 

ERC = enterprise RAG challenge

__GuardRails AI__ = a tool to validate LLM outputs. Can do:
- schema / types / values
- call LLM-as-a-judge
- custom Python checks

Check types:
- unit-tests
- human assessment
- A/B testing